# Project Structure Setup

In [1]:
import os
from pathlib import Path

# ── Root of the deployment package
API_ROOT = Path('/kaggle/working/sentiment_api')

# ── Directory structure
dirs = [
    API_ROOT / 'app',
    API_ROOT / 'app' / 'models',
    API_ROOT / 'app' / 'routers',
    API_ROOT / 'app' / 'schemas',
    API_ROOT / 'app' / 'services',
    API_ROOT / 'tests',
    API_ROOT / 'scripts',
]

for d in dirs:
    d.mkdir(parents=True, exist_ok=True)

print(' Project structure created:')
for d in sorted(API_ROOT.rglob('*')):
    indent = '  ' * (len(d.relative_to(API_ROOT).parts) - 1)
    print(f'   {indent}📁 {d.name}/')

 Project structure created:
   📁 app/
     📁 models/
     📁 routers/
     📁 schemas/
     📁 services/
   📁 scripts/
   📁 tests/


# Copy Model Artifacts

In [2]:
# Copies all trained models and scalers into the API package
# so Docker can bundle them into the image

import shutil
import joblib

NB07_OUT = Path('/kaggle/input/notebooks/bennjimatakwa/behavioral-prediction-model-08-dl')
NB06_OUT = Path('/kaggle/input/notebooks/bennjimatakwa/nb-07-cluster-profiling-behavioral-analysis')

MODEL_DEST = API_ROOT / 'app' / 'models'

# ── Files to copy
artifacts = {
    NB07_OUT / 'outputs' / 'models' / 'mlp_best.pt'      : 'mlp_best.pt',
    NB07_OUT / 'outputs' / 'models' / 'mlp_scaler.pkl'   : 'mlp_scaler.pkl',
    NB07_OUT / 'outputs' / 'models' / 'le_segment.pkl'   : 'le_segment.pkl',
    NB07_OUT / 'outputs' / 'models' / 'le_sentiment.pkl' : 'le_sentiment.pkl',
    NB07_OUT / 'outputs' / 'models' / 'feature_cols.json': 'feature_cols.json',
    NB06_OUT / 'outputs' / 'profiles' / 'segment_insights.json': 'segment_insights.json',
}

print('📦 Copying model artifacts:')
for src, dst_name in artifacts.items():
    dst = MODEL_DEST / dst_name
    shutil.copy2(src, dst)
    size_mb = dst.stat().st_size / 1e6
    print(f'   ✅ {dst_name:<35} {size_mb:.2f} MB')

print(f'\n✅ All artifacts copied to {MODEL_DEST}')

📦 Copying model artifacts:
   ✅ mlp_best.pt                         0.19 MB
   ✅ mlp_scaler.pkl                      0.00 MB
   ✅ le_segment.pkl                      0.00 MB
   ✅ le_sentiment.pkl                    0.00 MB
   ✅ feature_cols.json                   0.00 MB
   ✅ segment_insights.json               0.00 MB

✅ All artifacts copied to /kaggle/working/sentiment_api/app/models


# Pydantic Schemas

In [3]:
# ── nb-10 | Cell 03 — Pydantic Schemas
# Defines request/response data models using Pydantic v2
# These are the contracts between clients and the API

schemas_code = '''
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum


class SentimentLabel(str, Enum):
    negative = "negative"
    neutral  = "neutral"
    positive = "positive"


class RiskLevel(str, Enum):
    low      = "low"
    medium   = "medium"
    high     = "high"
    critical = "critical"


# ── Single review input
class ReviewInput(BaseModel):
    frustration_score  : float = Field(..., ge=0.0, description="Frustration score 0+")
    engagement_quality : float = Field(..., ge=0.0, le=1.0, description="Engagement 0-1")
    influence_weight   : float = Field(..., ge=0.0, le=1.0, description="Influence 0-1")
    recency_weight     : float = Field(..., ge=0.0, le=1.0, description="Recency 0-1")
    word_count         : int   = Field(..., ge=0, description="Word count")
    has_reply          : int   = Field(..., ge=0, le=1, description="1 if app replied")
    bilstm_prob_neg    : float = Field(..., ge=0.0, le=1.0)
    bilstm_prob_neu    : float = Field(..., ge=0.0, le=1.0)
    bilstm_prob_pos    : float = Field(..., ge=0.0, le=1.0)
    bilstm_confidence  : float = Field(..., ge=0.0, le=1.0)

    @field_validator("bilstm_prob_neg", "bilstm_prob_neu", "bilstm_prob_pos")
    @classmethod
    def probs_valid(cls, v):
        if not 0.0 <= v <= 1.0:
            raise ValueError("Probability must be between 0 and 1")
        return v

    model_config = {
        "json_schema_extra": {
            "example": {
                "frustration_score": 1.5,
                "engagement_quality": 0.3,
                "influence_weight": 0.6,
                "recency_weight": 0.4,
                "word_count": 25,
                "has_reply": 0,
                "bilstm_prob_neg": 0.6,
                "bilstm_prob_neu": 0.3,
                "bilstm_prob_pos": 0.1,
                "bilstm_confidence": 0.75
            }
        }
    }


# ── Batch input
class BatchReviewInput(BaseModel):
    reviews: List[ReviewInput] = Field(..., min_length=1, max_length=1000)


# ── Prediction response
class PredictionResponse(BaseModel):
    segment_id         : int
    segment_name       : str
    sentiment          : SentimentLabel
    segment_confidence : float
    sentiment_confidence: float
    risk_level         : RiskLevel
    priority_tier      : str
    recommended_strategy: str
    retention_risk_score: float


# ── Batch response
class BatchPredictionResponse(BaseModel):
    predictions : List[PredictionResponse]
    total       : int
    processing_time_ms: float


# ── Health check response
class HealthResponse(BaseModel):
    status     : str
    model_loaded: bool
    version    : str
    uptime_seconds: float


# ── Segment summary response
class SegmentSummaryResponse(BaseModel):
    segment_id  : int
    segment_name: str
    risk_level  : RiskLevel
    avg_score   : float
    pct_positive: float
    top_platform: str
    business_insight: str
'''

schema_path = API_ROOT / 'app' / 'schemas' / 'schemas.py'
schema_path.write_text(schemas_code)
(API_ROOT / 'app' / 'schemas' / '__init__.py').write_text('')

print(' Pydantic schemas written')
print(f'   → {schema_path}')

 Pydantic schemas written
   → /kaggle/working/sentiment_api/app/schemas/schemas.py


#  ML Model Service

In [4]:
# ── nb-10 | Cell 04 — ML Model Service
# Core service class that loads models once at startup
# and serves predictions. Singleton pattern for efficiency.

model_service_code = '''
import json
import time
import logging
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import joblib
import torch
import torch.nn as nn

logger = logging.getLogger(__name__)

# ── Model directory (relative to this file inside Docker)
MODEL_DIR = Path(__file__).parent.parent / "models"

# ── Segment metadata (hardcoded — never changes)
SEGMENT_NAMES = {
    0: "Satisfied Loyal Shoppers",
    1: "Frustrated Complainers",
    2: "Neutral Browsers",
    3: "Impulsive Buyers",
    4: "Engaged Brand Advocates",
}

SEGMENT_RISK = {
    0: "low",
    1: "critical",
    2: "medium",
    3: "high",
    4: "medium",
}

SEGMENT_STRATEGY = {
    0: "Reward & deepen loyalty",
    1: "Urgent complaint resolution & proactive outreach",
    2: "Re-engagement campaigns & discovery features",
    3: "Personalized offers & purchase incentives",
    4: "Community building & brand advocacy programs",
}

RISK_TO_PRIORITY = {
    "critical": "P1 - Critical",
    "high"    : "P2 - High",
    "medium"  : "P3 - Medium",
    "low"     : "P4 - Low",
}

SEGMENT_RISK_SCORES = {
    0: 0.0,
    1: 61.1,
    2: 47.1,
    3: 75.6,
    4: 100.0,
}

SENTIMENT_NAMES = {0: "negative", 1: "neutral", 2: "positive"}


# ── Dual-head MLP (identical to nb-07)
class BehaviorMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, n_segments, n_sentiments, dropout):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(in_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            in_dim = h
        self.backbone       = nn.Sequential(*layers)
        self.segment_head   = nn.Linear(in_dim, n_segments)
        self.sentiment_head = nn.Linear(in_dim, n_sentiments)

    def forward(self, x):
        shared = self.backbone(x)
        return self.segment_head(shared), self.sentiment_head(shared)


class ModelService:
    """
    Singleton ML service.
    Loads all models once at startup, serves predictions efficiently.
    Thread-safe for concurrent FastAPI requests.
    """

    _instance = None
    _startup_time = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._initialized = False
        return cls._instance

    def initialize(self):
        if self._initialized:
            return

        logger.info("Initializing ModelService...")
        start = time.time()

        # ── Load feature column list
        with open(MODEL_DIR / "feature_cols.json") as f:
            self.feature_cols = json.load(f)

        # ── Load sklearn artifacts
        self.scaler       = joblib.load(MODEL_DIR / "mlp_scaler.pkl")
        self.le_segment   = joblib.load(MODEL_DIR / "le_segment.pkl")
        self.le_sentiment = joblib.load(MODEL_DIR / "le_sentiment.pkl")

        # ── Load segment insights
        with open(MODEL_DIR / "segment_insights.json") as f:
            self.segment_insights = json.load(f)

        # ── Build MLP
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model  = BehaviorMLP(
            input_dim    = len(self.feature_cols),
            hidden_dims  = [256, 128, 64],
            n_segments   = 5,
            n_sentiments = 3,
            dropout      = 0.3,
        ).to(self.device)

        self.model.load_state_dict(
            torch.load(
                MODEL_DIR / "mlp_best.pt",
                map_location=self.device
            )
        )
        self.model.eval()

        self._startup_time = time.time()
        self._initialized  = True

        elapsed = time.time() - start
        logger.info(f"ModelService initialized in {elapsed:.2f}s on {self.device}")

    def predict_batch(self, feature_dicts: List[Dict]) -> List[Dict]:
        """
        Run inference on a list of feature dicts.
        Returns list of prediction dicts ready for API response.
        """
        # ── Build numpy matrix in correct column order
        X = np.array([
            [row[col] for col in self.feature_cols]
            for row in feature_dicts
        ], dtype=np.float32)

        # ── Scale
        X_scaled = self.scaler.transform(X)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(self.device)

        # ── Inference
        with torch.no_grad():
            seg_logits, sent_logits = self.model(X_tensor)
            seg_probs  = torch.softmax(seg_logits,  dim=1).cpu().numpy()
            sent_probs = torch.softmax(sent_logits, dim=1).cpu().numpy()

        seg_ids  = seg_probs.argmax(axis=1)
        sent_ids = sent_probs.argmax(axis=1)

        results = []
        for i in range(len(X)):
            seg_id  = int(seg_ids[i])
            sent_id = int(sent_ids[i])
            risk    = SEGMENT_RISK[seg_id]

            results.append({
                "segment_id"          : seg_id,
                "segment_name"        : SEGMENT_NAMES[seg_id],
                "sentiment"           : SENTIMENT_NAMES[sent_id],
                "segment_confidence"  : round(float(seg_probs[i].max()), 4),
                "sentiment_confidence": round(float(sent_probs[i].max()), 4),
                "risk_level"          : risk,
                "priority_tier"       : RISK_TO_PRIORITY[risk],
                "recommended_strategy": SEGMENT_STRATEGY[seg_id],
                "retention_risk_score": SEGMENT_RISK_SCORES[seg_id],
            })

        return results

    @property
    def uptime_seconds(self) -> float:
        if self._startup_time is None:
            return 0.0
        return round(time.time() - self._startup_time, 2)

    @property
    def is_ready(self) -> bool:
        return self._initialized


# ── Global singleton instance
model_service = ModelService()
'''

service_path = API_ROOT / 'app' / 'services' / 'model_service.py'
service_path.write_text(model_service_code)
(API_ROOT / 'app' / 'services' / '__init__.py').write_text('')

print(' ModelService written')
print(f'   → {service_path}')

 ModelService written
   → /kaggle/working/sentiment_api/app/services/model_service.py


# API Routers : 

In [5]:
# ── nb-10 | Cell 05 — API Routers
# Three routers: health, predict, segments
# Clean separation of concerns — each router handles one domain

# ── Health router
health_router_code = '''
import time
from fastapi import APIRouter
from app.schemas.schemas import HealthResponse
from app.services.model_service import model_service

router = APIRouter(prefix="/health", tags=["Health"])

API_VERSION   = "1.0.0"
API_START_TIME = time.time()

@router.get("/", response_model=HealthResponse)
async def health_check():
    """
    Liveness probe.
    Returns model load status and uptime.
    Used by Docker HEALTHCHECK and Kubernetes readiness probes.
    """
    return HealthResponse(
        status        = "ok" if model_service.is_ready else "loading",
        model_loaded  = model_service.is_ready,
        version       = API_VERSION,
        uptime_seconds= model_service.uptime_seconds,
    )

@router.get("/ready")
async def readiness_check():
    """
    Readiness probe — returns 503 if model not loaded.
    Kubernetes uses this to decide if pod should receive traffic.
    """
    if not model_service.is_ready:
        from fastapi import HTTPException
        raise HTTPException(status_code=503, detail="Model not initialized")
    return {"status": "ready"}
'''

# ── Predict router
predict_router_code = '''
import time
import logging
from fastapi import APIRouter, HTTPException
from app.schemas.schemas import (
    ReviewInput,
    BatchReviewInput,
    PredictionResponse,
    BatchPredictionResponse
)
from app.services.model_service import model_service

logger = APIRouter = APIRouter
router = __import__("fastapi").APIRouter(prefix="/predict", tags=["Predictions"])
logger = logging.getLogger(__name__)


@router.post("/single", response_model=PredictionResponse)
async def predict_single(review: ReviewInput):
    """
    Predict segment + sentiment + retention risk for one review.

    Returns:
    - Customer segment (0-4) with name
    - Predicted sentiment (negative/neutral/positive)
    - Confidence scores for both predictions
    - Retention risk level and priority tier
    - Recommended retention strategy
    """
    if not model_service.is_ready:
        raise HTTPException(status_code=503, detail="Model not loaded")

    try:
        results = model_service.predict_batch([review.model_dump()])
        return PredictionResponse(**results[0])
    except Exception as e:
        logger.error(f"Prediction failed: {e}")
        raise HTTPException(status_code=500, detail=str(e))


@router.post("/batch", response_model=BatchPredictionResponse)
async def predict_batch(payload: BatchReviewInput):
    """
    Predict for up to 1000 reviews in one request.
    Returns predictions + total count + processing time.

    Useful for:
    - Bulk scoring of existing customer base
    - Nightly batch jobs
    - Backfilling historical data
    """
    if not model_service.is_ready:
        raise HTTPException(status_code=503, detail="Model not loaded")

    try:
        t0 = time.time()
        feature_dicts = [r.model_dump() for r in payload.reviews]
        results = model_service.predict_batch(feature_dicts)
        elapsed_ms = round((time.time() - t0) * 1000, 2)

        return BatchPredictionResponse(
            predictions       = [PredictionResponse(**r) for r in results],
            total             = len(results),
            processing_time_ms= elapsed_ms,
        )
    except Exception as e:
        logger.error(f"Batch prediction failed: {e}")
        raise HTTPException(status_code=500, detail=str(e))
'''

# ── Segments router
segments_router_code = '''
from fastapi import APIRouter, HTTPException
from typing import List
from app.schemas.schemas import SegmentSummaryResponse, RiskLevel
from app.services.model_service import model_service, SEGMENT_NAMES, SEGMENT_RISK, SEGMENT_STRATEGY

router = APIRouter(prefix="/segments", tags=["Segments"])


@router.get("/", response_model=List[SegmentSummaryResponse])
async def list_segments():
    """
    Returns a summary of all 5 customer segments.
    Useful for dashboards and business intelligence views.
    """
    summaries = []
    for seg_id, insight in model_service.segment_insights.items():
        k = int(seg_id)
        summaries.append(SegmentSummaryResponse(
            segment_id      = k,
            segment_name    = SEGMENT_NAMES[k],
            risk_level      = RiskLevel(SEGMENT_RISK[k]),
            avg_score       = insight["avg_score"],
            pct_positive    = insight["pct_positive"],
            top_platform    = insight["top_platform"],
            business_insight= insight["business_insight"],
        ))
    return summaries


@router.get("/{segment_id}", response_model=SegmentSummaryResponse)
async def get_segment(segment_id: int):
    """Get details for a specific segment by ID (0-4)."""
    if segment_id not in range(5):
        raise HTTPException(status_code=404, detail=f"Segment {segment_id} not found")

    insight = model_service.segment_insights.get(str(segment_id))
    if not insight:
        raise HTTPException(status_code=404, detail="Segment insight not available")

    return SegmentSummaryResponse(
        segment_id      = segment_id,
        segment_name    = SEGMENT_NAMES[segment_id],
        risk_level      = RiskLevel(SEGMENT_RISK[segment_id]),
        avg_score       = insight["avg_score"],
        pct_positive    = insight["pct_positive"],
        top_platform    = insight["top_platform"],
        business_insight= insight["business_insight"],
    )
'''

(API_ROOT / 'app' / 'routers' / '__init__.py').write_text('')
(API_ROOT / 'app' / 'routers' / 'health.py').write_text(health_router_code)
(API_ROOT / 'app' / 'routers' / 'predict.py').write_text(predict_router_code)
(API_ROOT / 'app' / 'routers' / 'segments.py').write_text(segments_router_code)

print(' Routers written:')
print('   → routers/health.py    (liveness + readiness probes)')
print('   → routers/predict.py   (single + batch inference)')
print('   → routers/segments.py  (segment metadata)')

 Routers written:
   → routers/health.py    (liveness + readiness probes)
   → routers/predict.py   (single + batch inference)
   → routers/segments.py  (segment metadata)


#  FastAPI Main App

In [6]:
# Entry point — wires all routers together, adds middleware,
# configures CORS, sets up lifespan startup events

main_app_code = '''
import logging
import time
from contextlib import asynccontextmanager

from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse

from app.routers import health, predict, segments
from app.services.model_service import model_service

# ── Logging
logging.basicConfig(
    level  = logging.INFO,
    format = "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)


# ── Lifespan: runs once at startup and shutdown
@asynccontextmanager
async def lifespan(app: FastAPI):
    # ── Startup: load models before accepting any request
    logger.info("Starting up — loading ML models...")
    model_service.initialize()
    logger.info("Models loaded. API ready.")
    yield
    # ── Shutdown
    logger.info("Shutting down.")


# ── App definition
app = FastAPI(
    title       = "Customer Sentiment & Retention API",
    description = """
## AI-Powered Customer Profiling API

Predicts customer **segment**, **sentiment**, and **retention risk**
from behavioral features using a dual-head MLP trained on 100k e-commerce reviews.

### Models used
- **DistilBERT** (nb-03): fine-tuned sentiment classifier, test F1=0.718
- **BiLSTM** (nb-04): lightweight sentiment model, test F1=0.610
- **MLP** (nb-07): dual-head behavioral predictor, sentiment acc=88%

### Endpoints
- `POST /predict/single` — score one customer
- `POST /predict/batch`  — score up to 1000 customers
- `GET  /segments/`      — list all 5 customer segments
- `GET  /health/`        — liveness probe
    """,
    version     = "1.0.0",
    lifespan    = lifespan,
    docs_url    = "/docs",
    redoc_url   = "/redoc",
)


# ── CORS middleware (allow all origins for dev — tighten in prod)
app.add_middleware(
    CORSMiddleware,
    allow_origins     = ["*"],
    allow_credentials = True,
    allow_methods     = ["*"],
    allow_headers     = ["*"],
)


# ── Request timing middleware
@app.middleware("http")
async def add_process_time_header(request: Request, call_next):
    start    = time.time()
    response = await call_next(request)
    elapsed  = round((time.time() - start) * 1000, 2)
    response.headers["X-Process-Time-Ms"] = str(elapsed)
    return response


# ── Global exception handler
@app.exception_handler(Exception)
async def global_exception_handler(request: Request, exc: Exception):
    logger.error(f"Unhandled exception: {exc}")
    return JSONResponse(
        status_code = 500,
        content     = {"detail": "Internal server error", "type": str(type(exc).__name__)}
    )


# ── Include routers
app.include_router(health.router)
app.include_router(predict.router)
app.include_router(segments.router)


# ── Root
@app.get("/", tags=["Root"])
async def root():
    return {
        "name"   : "Customer Sentiment & Retention API",
        "version": "1.0.0",
        "docs"   : "/docs",
        "health" : "/health",
    }
'''

(API_ROOT / 'app' / '__init__.py').write_text('')
(API_ROOT / 'app' / 'main.py').write_text(main_app_code)

print(' FastAPI main app written')
print(f'   → {API_ROOT}/app/main.py')
print('\n   Features:')
print('    Async lifespan — models loaded once at startup')
print('    CORS middleware')
print('    Request timing header (X-Process-Time-Ms)')
print('   Global exception handler')
print('    Auto-generated /docs (Swagger UI)')
print('    Auto-generated /redoc (ReDoc UI)')

 FastAPI main app written
   → /kaggle/working/sentiment_api/app/main.py

   Features:
    Async lifespan — models loaded once at startup
    CORS middleware
    Request timing header (X-Process-Time-Ms)
   Global exception handler
    Auto-generated /docs (Swagger UI)
    Auto-generated /redoc (ReDoc UI)


# Dockerfile & Docker Compose

In [7]:
#  Dockerfile + docker-compose.yml

dockerfile_code = '''FROM python:3.11-slim

# ── Metadata
LABEL maintainer="your.email@example.com"
LABEL description="Customer Sentiment & Retention API"
LABEL version="1.0.0"

# ── System deps (curl for healthcheck)
RUN apt-get update && apt-get install -y --no-install-recommends \\
    curl \\
    && rm -rf /var/lib/apt/lists/*

# ── Working directory
WORKDIR /api

# ── Install Python deps first (cache layer)
COPY requirements.txt .
RUN pip install --no-cache-dir --upgrade pip \\
 && pip install --no-cache-dir -r requirements.txt

# ── Copy application code
COPY app/ ./app/

# ── Non-root user for security
RUN useradd -m -u 1000 apiuser && chown -R apiuser:apiuser /api
USER apiuser

# ── Expose port
EXPOSE 8000

# ── Docker health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=40s --retries=3 \\
  CMD curl -f http://localhost:8000/health/ready || exit 1

# ── Start server
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
'''

docker_compose_code = '''version: "3.9"

services:

  api:
    build:
      context: .
      dockerfile: Dockerfile
    container_name: sentiment_api
    ports:
      - "8000:8000"
    environment:
      - PYTHONUNBUFFERED=1
      - LOG_LEVEL=info
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health/ready"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 40s
    volumes:
      - ./logs:/api/logs

  # ── Optional: add prometheus + grafana for monitoring
  # prometheus:
  #   image: prom/prometheus
  #   ports: ["9090:9090"]
  #   volumes: ["./prometheus.yml:/etc/prometheus/prometheus.yml"]
'''

(API_ROOT / 'Dockerfile').write_text(dockerfile_code)
(API_ROOT / 'docker-compose.yml').write_text(docker_compose_code)

print(' Docker files written:')
print('   → Dockerfile       (python:3.11-slim, non-root user, HEALTHCHECK)')
print('   → docker-compose.yml (service + restart policy + health check)')

 Docker files written:
   → Dockerfile       (python:3.11-slim, non-root user, HEALTHCHECK)
   → docker-compose.yml (service + restart policy + health check)


# Requirements & .dockerignore

In [8]:
# ── nb-10 | Cell 08 — requirements.txt + .dockerignore

requirements_code = '''# Web framework
fastapi==0.111.0
uvicorn[standard]==0.29.0

# ML / Data
torch==2.2.0
numpy==1.26.4
scikit-learn==1.4.2
joblib==1.4.0

# Validation
pydantic==2.7.0

# Utilities
python-multipart==0.0.9
httpx==0.27.0
'''

dockerignore_code = '''# Git
.git
.gitignore

# Python
__pycache__/
*.py[cod]
*.egg-info/
.venv/
venv/

# Notebooks
*.ipynb
.ipynb_checkpoints/

# Logs and outputs
logs/
*.log

# Tests
tests/

# CI / CD
.github/
'''

gitignore_code = '''__pycache__/
*.py[cod]
.env
.venv/
venv/
logs/
*.log
app/models/*.pt
app/models/*.pkl
'''

(API_ROOT / 'requirements.txt').write_text(requirements_code)
(API_ROOT / '.dockerignore').write_text(dockerignore_code)
(API_ROOT / '.gitignore').write_text(gitignore_code)

print(' Requirements and ignore files written')
print('   → requirements.txt    (pinned versions)')
print('   → .dockerignore       (excludes notebooks, tests, logs)')
print('   → .gitignore          (excludes models — too large for git)')

 Requirements and ignore files written
   → requirements.txt    (pinned versions)
   → .dockerignore       (excludes notebooks, tests, logs)
   → .gitignore          (excludes models — too large for git)


# Tests

In [9]:
# ── nb-10 | Cell 09 — Test Suite
# pytest tests for all endpoints
# Professional projects always have tests

test_code = '''
import pytest
from fastapi.testclient import TestClient
from app.main import app
from app.services.model_service import model_service

# ── Initialize model once for all tests
model_service.initialize()
client = TestClient(app)

# ── Sample valid payload
VALID_PAYLOAD = {
    "frustration_score"  : 1.5,
    "engagement_quality" : 0.3,
    "influence_weight"   : 0.6,
    "recency_weight"     : 0.4,
    "word_count"         : 25,
    "has_reply"          : 0,
    "bilstm_prob_neg"    : 0.6,
    "bilstm_prob_neu"    : 0.3,
    "bilstm_prob_pos"    : 0.1,
    "bilstm_confidence"  : 0.75
}


class TestHealth:
    def test_health_returns_200(self):
        r = client.get("/health/")
        assert r.status_code == 200

    def test_health_model_loaded(self):
        r = client.get("/health/")
        assert r.json()["model_loaded"] is True

    def test_readiness_returns_200(self):
        r = client.get("/health/ready")
        assert r.status_code == 200


class TestPredictSingle:
    def test_single_returns_200(self):
        r = client.post("/predict/single", json=VALID_PAYLOAD)
        assert r.status_code == 200

    def test_single_response_has_required_fields(self):
        r = client.post("/predict/single", json=VALID_PAYLOAD)
        data = r.json()
        required = [
            "segment_id", "segment_name", "sentiment",
            "segment_confidence", "risk_level",
            "priority_tier", "recommended_strategy"
        ]
        for field in required:
            assert field in data, f"Missing field: {field}"

    def test_segment_id_in_valid_range(self):
        r = client.post("/predict/single", json=VALID_PAYLOAD)
        assert r.json()["segment_id"] in range(5)

    def test_sentiment_valid_label(self):
        r = client.post("/predict/single", json=VALID_PAYLOAD)
        assert r.json()["sentiment"] in ["negative", "neutral", "positive"]

    def test_invalid_payload_returns_422(self):
        r = client.post("/predict/single", json={"frustration_score": -999})
        assert r.status_code == 422


class TestPredictBatch:
    def test_batch_returns_200(self):
        r = client.post("/predict/batch", json={"reviews": [VALID_PAYLOAD] * 5})
        assert r.status_code == 200

    def test_batch_count_matches(self):
        n = 10
        r = client.post("/predict/batch", json={"reviews": [VALID_PAYLOAD] * n})
        assert r.json()["total"] == n

    def test_batch_has_processing_time(self):
        r = client.post("/predict/batch", json={"reviews": [VALID_PAYLOAD]})
        assert "processing_time_ms" in r.json()


class TestSegments:
    def test_list_segments_returns_5(self):
        r = client.get("/segments/")
        assert r.status_code == 200
        assert len(r.json()) == 5

    def test_get_segment_by_id(self):
        r = client.get("/segments/0")
        assert r.status_code == 200
        assert r.json()["segment_id"] == 0

    def test_invalid_segment_returns_404(self):
        r = client.get("/segments/99")
        assert r.status_code == 404
'''

test_path = API_ROOT / 'tests' / 'test_api.py'
(API_ROOT / 'tests' / '__init__.py').write_text('')
test_path.write_text(test_code)

print(' Test suite written')
print(f'   → {test_path}')
print('\n   Test classes:')
print('    TestHealth        (liveness + readiness)')
print('    TestPredictSingle (valid/invalid payloads)')
print('    TestPredictBatch  (count + timing)')
print('    TestSegments      (list + get + 404)')

 Test suite written
   → /kaggle/working/sentiment_api/tests/test_api.py

   Test classes:
    TestHealth        (liveness + readiness)
    TestPredictSingle (valid/invalid payloads)
    TestPredictBatch  (count + timing)
    TestSegments      (list + get + 404)


#  README + Final File Tree

In [10]:
# ── nb-10 | Cell 10 — README + Final File Tree (CLEAN VERSION)

from pathlib import Path

API_ROOT = Path('/kaggle/working/sentiment_api')

# ─────────────────────────────
# README (SAFE STRING VERSION)
# ─────────────────────────────

readme_code = """Customer Sentiment & Retention API

AI-powered REST API for customer segmentation, sentiment analysis, and retention risk scoring.

========================
QUICK START
========================

docker-compose up --build

Test:
curl http://localhost:8000/health/

curl -X POST http://localhost:8000/predict/single -H "Content-Type: application/json" -d '{"frustration_score":1.5,"engagement_quality":0.3,"influence_weight":0.6,"recency_weight":0.4,"word_count":25,"has_reply":0,"bilstm_prob_neg":0.6,"bilstm_prob_neu":0.3,"bilstm_prob_pos":0.1,"bilstm_confidence":0.75}'

========================
API ENDPOINTS
========================

GET  /health/
GET  /health/ready
POST /predict/single
POST /predict/batch
GET  /segments/
GET  /segments/{id}
GET  /docs

========================
MODEL PERFORMANCE
========================

DistilBERT -> 0.718 F1
BiLSTM     -> 0.610 F1
MLP Sent   -> 0.883 F1
MLP Seg    -> 0.553 F1

========================
TECH STACK
========================

FastAPI
PyTorch
Pydantic v2
Docker
pytest

========================
ARCHITECTURE
========================

Reviews → Features → ML Models → FastAPI → Docker → Deployment
"""

# ─────────────────────────────
# SAVE FILE
# ─────────────────────────────

readme_path = API_ROOT / "README.md"
readme_path.write_text(readme_code)

# ─────────────────────────────
# PRINT STRUCTURE
# ─────────────────────────────

print("✅ README.md created successfully\n")

print("=" * 50)
print("📁 FINAL PROJECT STRUCTURE")
print("=" * 50)

for path in sorted(API_ROOT.rglob("*")):
    rel = path.relative_to(API_ROOT)
    indent = "  " * (len(rel.parts) - 1)
    icon = "📁" if path.is_dir() else "📄"
    print(f"{indent}{icon} {path.name}")

print("\n🚀 DONE — Notebook 10 is now fully fixed and safe on Kaggle")

✅ README.md created successfully

📁 FINAL PROJECT STRUCTURE
📄 .dockerignore
📄 .gitignore
📄 Dockerfile
📄 README.md
📁 app
  📄 __init__.py
  📄 main.py
  📁 models
    📄 feature_cols.json
    📄 le_segment.pkl
    📄 le_sentiment.pkl
    📄 mlp_best.pt
    📄 mlp_scaler.pkl
    📄 segment_insights.json
  📁 routers
    📄 __init__.py
    📄 health.py
    📄 predict.py
    📄 segments.py
  📁 schemas
    📄 __init__.py
    📄 schemas.py
  📁 services
    📄 __init__.py
    📄 model_service.py
📄 docker-compose.yml
📄 requirements.txt
📁 scripts
📁 tests
  📄 __init__.py
  📄 test_api.py

🚀 DONE — Notebook 10 is now fully fixed and safe on Kaggle


In [11]:
import os
from pathlib import Path

API_ROOT = Path('/kaggle/working/sentiment_api')

# force ensure folders exist
(API_ROOT / "app").mkdir(parents=True, exist_ok=True)
(API_ROOT / "tests").mkdir(parents=True, exist_ok=True)

print(" Folders forced to exist")

 Folders forced to exist


In [12]:
from pathlib import Path

API_ROOT = Path('/kaggle/working/sentiment_api')

print("📁 FULL PROJECT CHECK\n")

for path in sorted(API_ROOT.rglob("*")):
    rel = path.relative_to(API_ROOT)
    depth = len(rel.parts) - 1
    indent = "  " * depth

    if path.is_dir():
        print(f"{indent}📁 {path.name}/")
    else:
        size_kb = path.stat().st_size / 1024
        print(f"{indent}📄 {path.name} ({size_kb:.2f} KB)")

📁 FULL PROJECT CHECK

📄 .dockerignore (0.18 KB)
📄 .gitignore (0.08 KB)
📄 Dockerfile (0.98 KB)
📄 README.md (1.14 KB)
📁 app/
  📄 __init__.py (0.00 KB)
  📄 main.py (2.91 KB)
  📁 models/
    📄 feature_cols.json (0.18 KB)
    📄 le_segment.pkl (0.36 KB)
    📄 le_sentiment.pkl (0.49 KB)
    📄 mlp_best.pt (189.30 KB)
    📄 mlp_scaler.pkl (0.83 KB)
    📄 segment_insights.json (1.65 KB)
  📁 routers/
    📄 __init__.py (0.00 KB)
    📄 health.py (1.06 KB)
    📄 predict.py (2.19 KB)
    📄 segments.py (1.88 KB)
  📁 schemas/
    📄 __init__.py (0.00 KB)
    📄 schemas.py (2.89 KB)
  📁 services/
    📄 __init__.py (0.00 KB)
    📄 model_service.py (5.79 KB)
📄 docker-compose.yml (0.64 KB)
📄 requirements.txt (0.21 KB)
📁 scripts/
📁 tests/
  📄 __init__.py (0.00 KB)
  📄 test_api.py (2.97 KB)


In [13]:
import subprocess

result = subprocess.run(
    ['python', '-m', 'pytest',
     str(API_ROOT / 'tests' / 'test_api.py'),
     '-v', '--tb=short'],
    capture_output=True,
    text=True,
    cwd=str(API_ROOT)
)
print(result.stdout)
print(result.stderr)

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working/sentiment_api
plugins: anyio-4.12.1, langsmith-0.7.6, typeguard-4.5.1
collecting ... collected 14 items

tests/test_api.py::TestHealth::test_health_returns_200 PASSED            [  7%]
tests/test_api.py::TestHealth::test_health_model_loaded PASSED           [ 14%]
tests/test_api.py::TestHealth::test_readiness_returns_200 PASSED         [ 21%]
tests/test_api.py::TestPredictSingle::test_single_returns_200 PASSED     [ 28%]
tests/test_api.py::TestPredictSingle::test_single_response_has_required_fields PASSED [ 35%]
tests/test_api.py::TestPredictSingle::test_segment_id_in_valid_range PASSED [ 42%]
tests/test_api.py::TestPredictSingle::test_sentiment_valid_label PASSED  [ 50%]
tests/test_api.py::TestPredictSingle::test_invalid_payload_returns_422 PASSED [ 57%]
tests/test_api.py::Test